In [ ]:
!pip install isodate -q

import json
import re
import isodate
import requests
from fractions import Fraction
from bs4 import BeautifulSoup
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. MATEMATİKSEL MALZEME ÖLÇEKLEME MOTORU

class MalzemeMotoru:
    @staticmethod
    def sayi_donustur(metin):
        try:
            if " " in metin:
                tam, kesir = metin.split(" ")
                return float(tam) + float(Fraction(kesir))
            if "/" in metin:
                return float(Fraction(metin))
            return float(metin)
        except Exception:
            return None

    @classmethod
    def olcekle(cls, malzeme_metni, carpan):
        if carpan == 1.0:
            return malzeme_metni

        desen = r'^(\d+\s+\d+/\d+|\d+/\d+|\d+(?:\.\d+)?)'
        eslesme = re.match(desen, malzeme_metni.strip())

        if eslesme:
            eski_sayi_str = eslesme.group(1)
            sayisal_deger = cls.sayi_donustur(eski_sayi_str)
            if sayisal_deger:
                yeni_deger = sayisal_deger * carpan
                formatli = f"{int(yeni_deger)}" if yeni_deger.is_integer() else f"{yeni_deger:.2f}".rstrip("0").rstrip(".")
                return re.sub(desen, formatli, malzeme_metni.strip(), count=1)
        return malzeme_metni


# 2. SCHEMA.ORG/RECIPE UYUMLU CLASS YAPISI

class Tarif:
    def __init__(self, name, ingredients, cook_time=None, prep_time=None,
                 total_time=None, recipe_yield=None, category=None,
                 cuisine=None, instructions=None, gorsel_url="", url=""):
        self.ad = name
        self.malzemeler = ingredients or []
        self.pisirme_suresi_raw = cook_time
        self.hazirlik_suresi_raw = prep_time
        self.toplam_sure_raw = total_time
        self.kisi_sayisi_str = self._parse_yield(recipe_yield)
        self.varsayilan_porsiyon = self._extract_servings_int(self.kisi_sayisi_str)
        self.kategori = category
        self.mutfak = cuisine
        self.talimatlar = instructions or []
        self.gorsel_url = gorsel_url
        self.url = url

        self.pisirme_suresi_dk = self._iso_to_minutes(self.pisirme_suresi_raw)
        self.hazirlik_suresi_dk = self._iso_to_minutes(self.hazirlik_suresi_raw)
        self.toplam_sure_dk = self._iso_to_minutes(self.toplam_sure_raw)

    @staticmethod
    def _extract_servings_int(yield_str):
        rakamlar = re.findall(r'\d+', str(yield_str))
        return int(rakamlar[0]) if rakamlar else 4

    @staticmethod
    def _iso_to_minutes(iso_duration):
        if not iso_duration:
            return None
        try:
            dur = isodate.parse_duration(iso_duration)
            return int(dur.total_seconds()) // 60
        except Exception:
            m = re.search(r'(\d+)\s*(?:dakika|min|m)', str(iso_duration), re.IGNORECASE)
            return int(m.group(1)) if m else None

    @staticmethod
    def _parse_yield(recipe_yield):
        if not recipe_yield:
            return "4 porsiyon"
        if isinstance(recipe_yield, list):
            return str(recipe_yield[0])
        return str(recipe_yield)

    @classmethod
    def from_schema_json(cls, data, backup_image="", url=""):
        raw_instructions = data.get("recipeInstructions", [])
        instructions = []
        if isinstance(raw_instructions, list):
            for step in raw_instructions:
                if isinstance(step, dict):
                    instructions.append(step.get("text", ""))
                elif isinstance(step, str):
                    instructions.append(step)
        elif isinstance(raw_instructions, str):
            instructions.append(raw_instructions)

        img_val = data.get("image")
        gorsel = ""
        if isinstance(img_val, str):
            gorsel = img_val
        elif isinstance(img_val, list) and len(img_val) > 0:
            first = img_val[0]
            gorsel = first if isinstance(first, str) else first.get("url", "")
        elif isinstance(img_val, dict):
            gorsel = img_val.get("url", "")

        if not gorsel:
            gorsel = backup_image

        return cls(
            name=data.get("name", "İsimsiz Tarif"),
            ingredients=data.get("recipeIngredient", []),
            cook_time=data.get("cookTime"),
            prep_time=data.get("prepTime"),
            total_time=data.get("totalTime"),
            recipe_yield=data.get("recipeYield"),
            category=data.get("recipeCategory"),
            cuisine=data.get("recipeCuisine"),
            instructions=instructions,
            gorsel_url=gorsel,
            url=url
        )

    def dinamik_malzeme_listesi(self, hedef_kisi):
        carpan = hedef_kisi / self.varsayilan_porsiyon
        return [MalzemeMotoru.olcekle(m, carpan) for m in self.malzemeler]

    def kiler_kontrolu(self, evdeki_malzemeler):
        if not evdeki_malzemeler:
            return 0, self.malzemeler, []

        evdekiler = [e.strip().lower() for e in evdeki_malzemeler if e.strip()]
        var_olanlar = []
        eksikler = []

        for m in self.malzemeler:
            m_lower = m.lower()
            if any(e in m_lower for e in evdekiler):
                var_olanlar.append(m)
            else:
                eksikler.append(m)

        oran = int((len(var_olanlar) / len(self.malzemeler)) * 100) if self.malzemeler else 0
        return oran, var_olanlar, eksikler

    def sure_bilgisi(self):
        if self.pisirme_suresi_dk:
            return f"{self.pisirme_suresi_dk} dakika (Pişirme)"
        elif self.toplam_sure_dk:
            return f"{self.toplam_sure_dk} dakika (Toplam)"
        elif self.hazirlik_suresi_dk:
            return f"{self.hazirlik_suresi_dk} dakika (Hazırlık)"
        return "Süre belirtilmemiş."

    def __repr__(self):
        return f"<Tarif: {self.ad} | Orijinal Porsiyon: {self.varsayilan_porsiyon}>"


# 3. SCRAPING VE VERİ ÇEKME FONKSİYONU

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

def bonappetit_tarif_cek(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=12)
        if resp.status_code != 200:
            return None

        soup = BeautifulSoup(resp.content, "html.parser")
        backup_img = ""
        og_img_tag = soup.find("meta", property="og:image")
        if og_img_tag and og_img_tag.get("content"):
            backup_img = og_img_tag["content"]

        for s in soup.find_all("script", type="application/ld+json"):
            if not s.string:
                continue
            try:
                parsed = json.loads(s.string)
                items = parsed if isinstance(parsed, list) else [parsed]
                for item in items:
                    for node in item.get("@graph", [item]):
                        node_type = node.get("@type", "")
                        if node_type == "Recipe" or (isinstance(node_type, list) and "Recipe" in node_type):
                            return Tarif.from_schema_json(node, backup_image=backup_img, url=url)
            except Exception:
                continue
    except Exception:
        pass
    return None


tarif_linkleri = [
    "https://www.bonappetit.com/recipe/classic-french-toast",
    "https://www.bonappetit.com/recipe/creamy-white-bean-soup-with-kale-and-parmesan",
    "https://www.bonappetit.com/recipe/pasta-al-limone",
    "https://www.bonappetit.com/recipe/chocolate-chip-cookies",
    "https://www.bonappetit.com/recipe/focaccia-bread",
    "https://www.bonappetit.com/recipe/shakshuka",
    "https://www.bonappetit.com/recipe/cast-iron-roast-chicken-with-crispy-potatoes",
    "https://www.bonappetit.com/recipe/classic-guacamole",
    "https://www.bonappetit.com/recipe/brown-butter-toffee-chocolate-chip-cookies",
    "https://www.bonappetit.com/recipe/creamy-pasta-with-crispy-mushrooms"
]

tarif_listesi = []
print("Tarifler Oluşturuluyor...\n")

for idx, link in enumerate(tarif_linkleri, start=1):
    t = bonappetit_tarif_cek(link)
    if t:
        var_adi = f"tarif{idx}"
        globals()[var_adi] = t  # tarif1, tarif2, ..., tarif10 global değişkeni olur
        tarif_listesi.append(t)
        print(f"✓ [{var_adi}] {t.ad[:35]:<35} | {t.sure_bilgisi():<22} | Porsiyon: {t.varsayilan_porsiyon} kişi")

print(f"\n {len(tarif_listesi)} Tarif başarıyla hazırlandı!")


# UI/UX:

header_html = widgets.HTML("""
<div style="background: linear-gradient(135deg, #0f172a, #1e293b); color: #f8fafc; padding: 20px; border-radius: 12px; margin-bottom: 15px; border-left: 6px solid #f97316;">
    <h2 style="margin: 0 0 6px 0; font-size: 22px; font-family: sans-serif;">🍳 10 Niş Tarif: Akıllı Porsiyon & Kiler Asistanı</h2>
    <p style="margin: 0; font-size: 13px; color: #94a3b8;">Kişi sayısına göre gramajları otomatik yeniden ölçekleyin, evdeki malzemelerle uyum oranını hesaplayın.</p>
</div>
""")

dropdown_yemek = widgets.Dropdown(
    options=[t.ad for t in tarif_listesi],
    description="Tarif Seç:",
    layout=widgets.Layout(width="50%")
)

slider_kisi = widgets.IntSlider(
    value=4,
    min=1,
    max=16,
    step=1,
    description="Kişi Sayısı:",
    continuous_update=False,
    layout=widgets.Layout(width="45%")
)

input_dolap = widgets.Text(
    placeholder="Örn: egg, butter, garlic, olive oil (virgülle ayırın)",
    description="Kiler/Dolap:",
    layout=widgets.Layout(width="70%")
)

btn_hesapla = widgets.Button(
    description="Ölçekle & Analiz Et",
    button_style="warning",
    icon="calculator",
    layout=widgets.Layout(width="26%")
)

cikti_paneli = widgets.Output()

def tarif_degisti(change):
    secilen = next((t for t in tarif_listesi if t.ad == change.new), None)
    if secilen:
        slider_kisi.value = secilen.varsayilan_porsiyon

dropdown_yemek.observe(tarif_degisti, names='value')

def calistir(b):
    with cikti_paneli:
        cikti_paneli.clear_output()
        secilen = next((t for t in tarif_listesi if t.ad == dropdown_yemek.value), None)
        if not secilen:
            return

        hedef_kisi = slider_kisi.value
        evdeki_girdiler = [x.strip() for x in input_dolap.value.split(",") if x.strip()]

        olcekli_malzemeler = secilen.dinamik_malzeme_listesi(hedef_kisi)
        oran, bulunanlar, eksikler = secilen.kiler_kontrolu(evdeki_girdiler)

        img_html = f'<img src="{secilen.gorsel_url}" style="width: 230px; height: 180px; object-fit: cover; border-radius: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">' if secilen.gorsel_url else ''

        malzemeler_li = ""
        for m in olcekli_malzemeler:
            m_lower = m.lower()
            evde_var = any(e.lower() in m_lower for e in evdeki_girdiler) if evdeki_girdiler else False
            if evde_var:
                malzemeler_li += f"<li style='color: #15803d; font-weight: 600; margin-bottom: 4px;'>✓ {m} <small style='color: #166534; background: #dcfce7; padding: 1px 6px; border-radius: 4px;'>Dolapta Var</small></li>"
            else:
                malzemeler_li += f"<li style='color: #334155; margin-bottom: 4px;'>• {m}</li>"

        rozet = ""
        if evdeki_girdiler:
            renk = '#16a34a' if oran >= 70 else '#f59e0b' if oran >= 40 else '#ef4444'
            rozet = f"""
            <div style="background: #f8fafc; border: 1px solid #e2e8f0; border-radius: 8px; padding: 10px; margin-top: 10px;">
                <div style="display: flex; justify-content: space-between; align-items: center; font-size: 13px;">
                    <span style="font-weight: 600; color: #1e293b;">Kiler Uyumluluğu: %{oran}</span>
                    <span style="color: #64748b;">{len(bulunanlar)} hazır / {len(eksikler)} eksik</span>
                </div>
                <div style="width: 100%; background: #e2e8f0; border-radius: 6px; height: 7px; margin-top: 6px;">
                    <div style="width: {oran}%; background: {renk}; height: 7px; border-radius: 6px;"></div>
                </div>
            </div>
            """

        kart_html = f"""
        <div style="background: #ffffff; border: 1px solid #e2e8f0; border-radius: 12px; padding: 20px; margin-top: 15px; font-family: sans-serif; box-shadow: 0 4px 12px rgba(0,0,0,0.04);">
            <div style="display: flex; gap: 20px; align-items: flex-start;">
                {img_html}
                <div style="flex: 1;">
                    <h3 style="margin: 0; color: #0f172a; font-size: 21px;">{secilen.ad}</h3>
                    <div style="margin: 10px 0; display: flex; gap: 10px; flex-wrap: wrap;">
                        <span style="background: #e0f2fe; color: #0369a1; padding: 5px 12px; border-radius: 16px; font-size: 12px; font-weight: bold;">
                            {secilen.sure_bilgisi()}
                        </span>
                        <span style="background: #fef3c7; color: #92400e; padding: 5px 12px; border-radius: 16px; font-size: 12px; font-weight: bold;">
                            👥 Ayarlanan: {hedef_kisi} Kişilik (Orijinal: {secilen.varsayilan_porsiyon})
                        </span>
                    </div>
                    {rozet}
                </div>
            </div>

            <div style="margin-top: 18px; border-top: 1px solid #f1f5f9; padding-top: 14px;">
                <h4 style="margin: 0 0 10px 0; color: #1e293b; font-size: 15px;">
                    🛒 {hedef_kisi} Kişilik Porsiyona Göre Güncellenmiş Malzemeler:
                </h4>
                <ul style="padding-left: 20px; line-height: 1.6; margin: 0; max-height: 250px; overflow-y: auto;">
                    {malzemeler_li}
                </ul>
            </div>
        </div>
        """
        display(HTML(kart_html))

btn_hesapla.on_click(calistir)

if tarif_listesi:
    slider_kisi.value = tarif_listesi[0].varsayilan_porsiyon

ui = widgets.VBox([
    header_html,
    widgets.HBox([dropdown_yemek, slider_kisi]),
    widgets.HBox([input_dolap, btn_hesapla]),
    cikti_paneli
])

display(ui)

Tarifler Oluşturuluyor...

✓ [tarif1] BA's Best French Toast              | 10 dakika (Pişirme)    | Porsiyon: 4 kişi
✓ [tarif3] Creamy Lemon Pasta (Pasta al Limone | 15 dakika (Pişirme)    | Porsiyon: 4 kişi
✓ [tarif4] Chocolate Chip Cookies              | 20 dakika (Pişirme)    | Porsiyon: 20 kişi
✓ [tarif5] Focaccia Bread                      | Süre belirtilmemiş.    | Porsiyon: 18 kişi
✓ [tarif7] Cast-Iron Roast Chicken With Crispy | 45 dakika (Toplam)     | Porsiyon: 4 kişi
✓ [tarif10] Creamy Mushroom Pasta               | 25 dakika (Pişirme)    | Porsiyon: 4 kişi

 6 Tarif başarıyla hazırlandı!
